# 21.08 - Mask R-CNN ResNet50-FPN V2 from-scratch training

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** An offline Mask R-CNN V2 model, one full-fold training epoch, loss evidence, and cached validation predictions.

## Why this model

Torchvision 0.28 exposes two official Mask R-CNN builders. The V2 weights report **41.8 COCO mask mAP** versus **34.6** for the older builder, so V2 is the accuracy-first allowlisted baseline—not a universal winner. V2 is larger and more expensive (about 46.4M parameters and 333.6 GFLOPS in official metadata).

The default below is strictly offline: `weights=None` and `weights_backbone=None`. `use_pretrained=True` may download a 177 MB COCO checkpoint and is valid only if it is cached/attached and its training data is legal for the competition.

Training returns five losses: classifier, box regression, mask, RPN objectness, and RPN box regression. Evaluation returns `boxes`, `labels`, `scores`, and soft `masks [N,1,H,W]`.

In [ ]:
import csv
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import tv_tensors
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import masks_to_boxes
from torchvision.transforms import v2

SEED = 21
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "_day21_instance_data"
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.csv")
FOLD_PATH = os.path.join(DATA_DIR, "folds.csv")
NUM_CLASSES = 3  # background + circle + square
MODEL_POLICY = {"builder": "maskrcnn_resnet50_fpn_v2", "default_weights": None, "default_backbone_weights": None}
print("device:", DEVICE, "policy:", MODEL_POLICY)


## Prepared fold and Dataset

These helpers repeat the Day 20 data boundary so this notebook is self-contained.

**Return structure — `generate_instance_fixture`:** A `dict` with `data_dir` (`str`), `manifest_path` (`str`), `images` (`int`), and `strata` (`int`). It writes RGB PNGs, instance-ID mask PNGs, and `manifest.csv`.

In [ ]:
def generate_instance_fixture(data_dir, samples_per_stratum=6, image_size=96, seed=20):
    image_dir = os.path.join(data_dir, "images")
    mask_dir = os.path.join(data_dir, "masks")
    manifest_path = os.path.join(data_dir, "manifest.csv")
    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    rows = []
    sample_index = 0

    for split_key in ("circle_1", "square_1", "mixed_2"):
        for local_index in range(int(samples_per_stratum)):
            image_id = f"image_{sample_index:03d}"
            pixels = rng.integers(18, 36, size=(image_size, image_size, 3), dtype=np.uint8)
            image = Image.fromarray(pixels, mode="RGB")
            instance_map = Image.fromarray(np.zeros((image_size, image_size), dtype=np.uint8), mode="L")
            image_draw = ImageDraw.Draw(image)
            mask_draw = ImageDraw.Draw(instance_map)
            labels = []

            if split_key in ("circle_1", "mixed_2"):
                offset = int(local_index % 5)
                circle_box = (8 + offset, 10, 38 + offset, 40)
                image_draw.ellipse(circle_box, fill=(220, 70, 70), outline=(255, 220, 220), width=2)
                mask_draw.ellipse(circle_box, fill=1)
                labels.append(1)

            if split_key in ("square_1", "mixed_2"):
                offset = int((local_index * 3) % 7)
                square_box = (53 - offset, 51, 84 - offset, 82)
                instance_id = 1 if split_key == "square_1" else 2
                image_draw.rectangle(square_box, fill=(70, 170, 235), outline=(220, 245, 255), width=2)
                mask_draw.rectangle(square_box, fill=instance_id)
                labels.append(2)

            image_path = os.path.join(image_dir, image_id + ".png")
            mask_path = os.path.join(mask_dir, image_id + ".png")
            image.save(image_path)
            instance_map.save(mask_path)
            rows.append(
                {
                    "sample_index": sample_index,
                    "image_id": image_id,
                    "image_path": image_path,
                    "mask_path": mask_path,
                    "instance_labels": json.dumps(labels),
                    "split_key": split_key,
                }
            )
            sample_index += 1

    with open(manifest_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return {
        "data_dir": data_dir,
        "manifest_path": manifest_path,
        "images": len(rows),
        "strata": 3,
    }

fixture_info = generate_instance_fixture(DATA_DIR, samples_per_stratum=6, seed=SEED)
print(fixture_info)


**Return structure — `build_fold_table`:** A `pandas.DataFrame` with all manifest columns plus integer `fold`; one stable `image_id` appears exactly once. The same table is written to `output_path`.

In [ ]:
def build_fold_table(manifest_path, output_path, n_splits=3, seed=20):
    frame = pd.read_csv(manifest_path).sort_values("image_id").reset_index(drop=True)
    frame["fold"] = -1
    splitter = StratifiedKFold(n_splits=int(n_splits), shuffle=True, random_state=int(seed))
    for fold, (_, validation_indices) in enumerate(
        splitter.split(np.zeros(len(frame), dtype=np.uint8), frame["split_key"])
    ):
        frame.loc[validation_indices, "fold"] = int(fold)
    frame["fold"] = frame["fold"].astype(int)
    frame.to_csv(output_path, index=False)
    return frame

fold_frame = build_fold_table(MANIFEST_PATH, FOLD_PATH, n_splits=3, seed=SEED)
print(pd.crosstab(fold_frame["fold"], fold_frame["split_key"]))


**Return structure — `InstanceMaskDataset`:** A callable Dataset. `dataset[i]` returns `(image, target)`: image is CPU `float32 [3,H,W]` in `[0,1]`; target contains `boxes` (`float32 [N,4]`), `labels` (`int64 [N]`), `masks` (`uint8 [N,H,W]`), `image_id` (`int64 [1]`), `area` (`float32 [N]`), and `iscrowd` (`int64 [N]`).

In [ ]:
class InstanceMaskDataset(Dataset):
    def __init__(self, fold_frame, validation_fold, split, augment=False):
        if split not in ("train", "validation"):
            raise ValueError("split must be 'train' or 'validation'")
        selector = fold_frame["fold"] != int(validation_fold)
        if split == "validation":
            selector = ~selector
        self.rows = fold_frame.loc[selector].sort_values("image_id").reset_index(drop=True)
        operations = [v2.RandomHorizontalFlip(p=0.5)] if augment else []
        operations.extend([v2.ToDtype(torch.float32, scale=True), v2.ToPureTensor()])
        self.transforms = v2.Compose(operations)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[int(index)]
        image_array = np.array(Image.open(row["image_path"]).convert("RGB"), copy=True)
        instance_array = np.array(Image.open(row["mask_path"]), copy=True)
        instance_ids = [int(value) for value in np.unique(instance_array) if int(value) != 0]
        masks = torch.stack(
            [torch.from_numpy((instance_array == instance_id).astype(np.uint8)) for instance_id in instance_ids]
        )
        labels = torch.tensor(json.loads(row["instance_labels"]), dtype=torch.int64)
        boxes = masks_to_boxes(masks)
        image = tv_tensors.Image(torch.from_numpy(image_array).permute(2, 0, 1))
        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=instance_array.shape),
            "labels": labels,
            "masks": tv_tensors.Mask(masks),
            "image_id": torch.tensor([int(row["sample_index"])], dtype=torch.int64),
            "area": masks.flatten(1).sum(1).to(torch.float32),
            "iscrowd": torch.zeros(len(instance_ids), dtype=torch.int64),
        }
        image, target = self.transforms(image, target)
        target["boxes"] = target["boxes"].to(torch.float32)
        target["masks"] = target["masks"].to(torch.uint8)
        return image, target


**Return structure — `instance_collate`:** A tuple `(images, targets)`; each position is a tuple of length `B`, preserving variable image sizes and instance counts.

In [ ]:
def instance_collate(batch):
    images, targets = zip(*batch)
    return tuple(images), tuple(targets)

generator = torch.Generator().manual_seed(SEED)
train_dataset = InstanceMaskDataset(fold_frame, validation_fold=0, split="train", augment=True)
validation_dataset = InstanceMaskDataset(fold_frame, validation_fold=0, split="validation", augment=False)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, generator=generator, collate_fn=instance_collate)
validation_loader = DataLoader(validation_dataset, batch_size=2, shuffle=False, collate_fn=instance_collate)
print("train/validation images:", len(train_dataset), len(validation_dataset))


## Exercise 21-A: Build the official accuracy-first baseline

Use the V2 builder directly. For random initialization, specify both weight arguments so no backbone download occurs. The compact proposal limits shorten this toy run; retune them for real object sizes and compute budgets.

**Return structure — `build_maskrcnn_v2`:** A Torchvision `MaskRCNN` module on `device`. Its box and mask predictors each expose `num_classes` outputs including background. The default call uses no detector or backbone checkpoint. `use_pretrained=True` may download official weights and then replaces both predictors.

In [ ]:
def build_maskrcnn_v2(num_classes=3, use_pretrained=False, device=DEVICE):
    # TODO 21-A: build the V2 model; default to weights=None and weights_backbone=None.
    # If use_pretrained=True, replace both the box and mask predictors for num_classes.
    raise NotImplementedError

# Smoke check: construction must be offline and expose three classes.
model = build_maskrcnn_v2(num_classes=NUM_CLASSES, use_pretrained=False)
print(model.roi_heads.box_predictor.cls_score.out_features, model.roi_heads.mask_predictor.mask_fcn_logits.out_channels)


## Exercise 21-B: Train on the complete fold

The smoke experiment uses every training image once. Keep loss components separate: a falling total can hide a broken mask branch.

**Return structure — `train_one_epoch`:** A `dict` with Python-float means for `loss_classifier`, `loss_box_reg`, `loss_mask`, `loss_objectness`, `loss_rpn_box_reg`, and `total`; integer `batches` and `images`; and float `runtime_seconds`. The model parameters are updated.

In [ ]:
def train_one_epoch(model, loader, optimizer, device=DEVICE, max_batches=None):
    # TODO 21-B: sum the five losses, update the model, and return averaged components plus counts/runtime.
    raise NotImplementedError

# Smoke check and decision evidence: one complete, untruncated training fold.
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.9, weight_decay=0.0005)
training_stats = train_one_epoch(model, train_loader, optimizer, device=DEVICE, max_batches=None)
print(pd.DataFrame([training_stats]).round(4).to_string(index=False))


## Exercise 21-C: Cache validation inference once

Threshold sweeps belong after inference. Cache raw low-threshold predictions on CPU, keyed by stable ID, so Day 22 can tune without repeated model calls.

**Return structure — `cache_validation_predictions`:** A tuple `(records, seconds_per_image)`. `records` is a list with one dictionary per processed image: integer `image_id`; CPU `boxes` (`float32 [P,4]`), `labels` (`int64 [P]`), `scores` (`float32 [P]`), `masks` (`float32 [P,1,H,W]`), `gt_labels` (`int64 [G]`), and `gt_masks` (`uint8 [G,H,W]`). Position 1 is a Python float.

In [ ]:
def cache_validation_predictions(model, loader, device=DEVICE, max_images=None):
    # TODO 21-C: run eval/inference_mode once and cache CPU predictions plus ground truth by stable image ID.
    raise NotImplementedError

# Smoke check: cache the complete untouched validation fold.
validation_records, seconds_per_image = cache_validation_predictions(model, validation_loader, device=DEVICE)
prediction_support = [int(record["labels"].numel()) for record in validation_records]
print("validation records:", len(validation_records), "predictions/image:", prediction_support, "seconds/image:", round(seconds_per_image, 4))


## Test Cases

Tests enforce the offline default, class heads, complete-fold training evidence, five finite losses, and the standard prediction schema. They do not require a randomly initialized one-epoch model to be accurate.

**Return structure — `run_day21_tests`:** Returns `None`; assertions communicate failure and `Day 21 tests passed` communicates success.

In [ ]:
def run_day21_tests():
    assert MODEL_POLICY["default_weights"] is None and MODEL_POLICY["default_backbone_weights"] is None
    assert model.roi_heads.box_predictor.cls_score.out_features == NUM_CLASSES
    assert model.roi_heads.mask_predictor.mask_fcn_logits.out_channels == NUM_CLASSES
    expected_losses = {"loss_classifier", "loss_box_reg", "loss_mask", "loss_objectness", "loss_rpn_box_reg"}
    assert expected_losses <= set(training_stats)
    assert training_stats["images"] == len(train_dataset) and training_stats["batches"] == len(train_loader)
    assert all(np.isfinite(training_stats[name]) and training_stats[name] >= 0.0 for name in expected_losses)
    assert len(validation_records) == len(validation_dataset) == 6
    assert len({record["image_id"] for record in validation_records}) == 6
    for record in validation_records:
        assert set(record) == {"image_id", "boxes", "labels", "scores", "masks", "gt_labels", "gt_masks"}
        predictions = int(record["labels"].numel())
        assert record["boxes"].shape == (predictions, 4)
        assert record["scores"].shape == (predictions,)
        assert record["masks"].shape[:2] == (predictions, 1)
        assert record["gt_masks"].dtype == torch.uint8
    print("Day 21 tests passed")

run_day21_tests()


## Day 21 Checklist

- [ ] Record why V2 is the accuracy-first allowlisted baseline and its compute cost.
- [ ] Confirm the default performs no checkpoint download.
- [ ] Train on every image in the selected training fold.
- [ ] Inspect all five loss components, shape, dtype, device, and labels.
- [ ] Cache every validation image exactly once before threshold tuning.
- [ ] For real OOF training, repeat from a fresh seed-aligned initialization for every fold and save fold-specific checkpoints.
- [ ] Run the test cases.